In [1]:
import numpy as np
import cv2
import glob
import matplotlib.pyplot as plt
import pyrealsense2 as rs

In [2]:
# stream rgb and depth from L515 and save it. Press S to save and Q to quit

#from networkx import config
import pyrealsense2 as rs
import numpy as np
import cv2
import time
import os
from datetime import datetime

def realsense_stream_with_save():
    # Create save directory if it doesn't exist
    save_dir = "captured_images"
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        print(f"Created directory: {save_dir}")
    
    # Configure depth and color streams
    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
    config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
    # Enable color stream with 1920x1080 resolution
    # config.enable_stream(rs.stream.color, 1920, 1080, rs.format.bgr8, 30)
    
    # Enable depth stream for L515
    # config.enable_stream(rs.stream.depth, 1024, 768, rs.format.z16, 30)
    
    # Start streaming
    pipeline.start(config)
    
    # Counter for saved images
    save_counter = 0
    
    try:
        while True:
            # Wait for a coherent pair of frames
            frames = pipeline.wait_for_frames()
            color_frame = frames.get_color_frame()
            depth_frame = frames.get_depth_frame()
            
            if not color_frame:
                continue
            
            # Convert images to numpy arrays
            color_image = np.asanyarray(color_frame.get_data())
            if depth_frame:
                depth_image = np.asanyarray(depth_frame.get_data())
                # Apply colormap to depth for visualization
                depth_colormap = cv2.applyColorMap(
                    cv2.convertScaleAbs(depth_image, alpha=0.03), 
                    cv2.COLORMAP_JET
                )
            
            # Display the color image
            cv2.namedWindow('RealSense L515 - 1920x1080', cv2.WINDOW_NORMAL)
            
            # Add instructions to the image
            instructions = "Press 'S' to save, 'Q' to quit"
            cv2.putText(color_image, instructions, (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            
            cv2.imshow('RealSense L515 - 1920x1080', color_image)
            
            # Display depth image if available
            if depth_frame:
                cv2.namedWindow('RealSense L515 - Depth', cv2.WINDOW_NORMAL)
                cv2.imshow('RealSense L515 - Depth', depth_colormap)
            
            # Handle key presses
            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('q') or key == 27:  # 'q' or ESC
                break
            elif key == ord('s'):  # 's' to save image
                save_counter += 1
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:-3]
                
                # Generate filenames
                color_filename = f"{save_dir}/color_{timestamp}_{save_counter:04d}.png"
                depth_filename = f"{save_dir}/depth_{timestamp}_{save_counter:04d}.png"
                
                # Save color image
                cv2.imwrite(color_filename, color_image)
                print(f"Saved color image: {color_filename}")
                
                # Save depth image if available
                if depth_frame:
                    cv2.imwrite(depth_filename, depth_colormap)
                    print(f"Saved depth image: {depth_filename}")
                
                # Show confirmation on screen
                confirmation = f"Saved image #{save_counter}"
                temp_image = color_image.copy()
                cv2.putText(temp_image, confirmation, (10, 70), 
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                cv2.imshow('RealSense L515 - 1920x1080', temp_image)
                cv2.waitKey(500)  # Show confirmation for 500ms
                
    finally:
        # Stop streaming
        pipeline.stop()
        cv2.destroyAllWindows()
        print(f"Total images saved: {save_counter}")
        print(f"Images saved in: {os.path.abspath(save_dir)}")

if __name__ == "__main__":
    realsense_stream_with_save()

RuntimeError: No device connected

In [13]:
# Cell 1: camera calibration from the saved 640x480 colour images.
# This cell is self-contained. Delete the old calibration cells so that nothing overwrites C.npz.

import glob
import cv2
import numpy as np

# ---------- Settings ----------
IMAGE_PATTERN = r'color_*.png'       # calibration images in the notebook's folder (must only match calibration images)
BOARD_SIZE = (9, 6)                  # number of INNER corners (columns, rows)
SQUARE_SIZE_M = 0.027                # measured side length of one square in meters
IMAGE_WIDTH, IMAGE_HEIGHT = 640, 480 # resolution of the video the matrix is used for
EXCLUDE = []                         # image names to leave out, e.g. ['color_07.png'], filled in after looking at the error per image
FIX_ASPECT_RATIO = True              # forces fx = fy
FIX_PRINCIPAL_POINT = True           # keeps (cx, cy) at the image centre during the optimisation

# Termination criteria for the sub-pixel corner refinement
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# 3D corner positions of the board in its own frame (z = 0), in meters
objp = np.zeros((BOARD_SIZE[0] * BOARD_SIZE[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:BOARD_SIZE[0], 0:BOARD_SIZE[1]].T.reshape(-1, 2) * SQUARE_SIZE_M

# ---------- Find the corners in every image ----------
image_names = [name for name in sorted(glob.glob(IMAGE_PATTERN)) if name not in EXCLUDE]
print(len(image_names), "images found")

objpoints = []        # 3D board points, one array per usable image
imgpoints = []        # matching 2D corner pixels
used_image_names = []

for image_name in image_names:
    image = cv2.imread(image_name)
    if image is None or image.shape[:2] != (IMAGE_HEIGHT, IMAGE_WIDTH):
        print("Skipped (unreadable or wrong size):", image_name, None if image is None else image.shape)
        continue
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    board_found, corners = cv2.findChessboardCorners(gray, BOARD_SIZE, None)
    if board_found:
        objpoints.append(objp)
        imgpoints.append(cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria))
        used_image_names.append(image_name)
    else:
        print("No corners found in", image_name)

assert len(objpoints) > 0, "No images with a detected board - check IMAGE_PATTERN and BOARD_SIZE"
print("Calibration images used:", len(objpoints), "of", len(image_names))

# ---------- Calibration ----------
if FIX_ASPECT_RATIO or FIX_PRINCIPAL_POINT:
    init_matrix = np.array([[600.0, 0, IMAGE_WIDTH / 2], [0, 600.0, IMAGE_HEIGHT / 2], [0, 0, 1]])   # start guess; the centre is kept if FIX_PRINCIPAL_POINT
    flags = cv2.CALIB_USE_INTRINSIC_GUESS
    if FIX_ASPECT_RATIO:
        flags |= cv2.CALIB_FIX_ASPECT_RATIO
    if FIX_PRINCIPAL_POINT:
        flags |= cv2.CALIB_FIX_PRINCIPAL_POINT
else:
    init_matrix, flags = None, 0
rms, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, (IMAGE_WIDTH, IMAGE_HEIGHT), init_matrix, None, flags=flags)

# ---------- Quality check ----------
errors = []
for i, name in enumerate(used_image_names):
    projected, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], mtx, dist)     # where the corners should be
    errors.append(cv2.norm(imgpoints[i].reshape(-1, 2).astype(np.float64),
                           projected.reshape(-1, 2).astype(np.float64), cv2.NORM_L2) / np.sqrt(len(projected)))
    print(f"{name}: {errors[-1]:.2f} px")
np.set_printoptions(precision=3, suppress=True)
print(f"\nRMS reprojection error: {rms:.3f} px (good: below about 1 px)")
print("Worst image:", used_image_names[int(np.argmax(errors))], f"{max(errors):.2f} px")
print(f"fx/fy = {mtx[0, 0] / mtx[1, 1]:.3f} (should be close to 1)")
print("M_int =\n", mtx)
print("\nCopy into Q1_Analysis.py:")
print(f"focalXInt     = {mtx[0, 0]:.6f}")
print(f"focalYInt     = {mtx[1, 1]:.6f}")
print(f"principalXInt = {mtx[0, 2]:.6f}")
print(f"principalYInt = {mtx[1, 2]:.6f}")

20 images found
Calibration images used: 20 of 20
color_20261024_131348_362_0001.png: 2.82 px
color_20261024_131351_457_0002.png: 4.43 px
color_20261024_131354_993_0003.png: 2.31 px
color_20261024_131357_891_0004.png: 2.65 px
color_20261024_131359_855_0005.png: 3.46 px
color_20261024_131401_647_0006.png: 4.16 px
color_20261024_131403_871_0007.png: 3.17 px
color_20261024_131406_764_0008.png: 5.60 px
color_20261024_131409_500_0009.png: 3.68 px
color_20261024_131412_370_0010.png: 2.99 px
color_20261024_131415_795_0011.png: 2.86 px
color_20261024_131418_161_0012.png: 3.89 px
color_20261024_131420_291_0013.png: 3.43 px
color_20261024_131422_290_0014.png: 3.45 px
color_20261024_131424_420_0015.png: 2.86 px
color_20261024_131428_650_0016.png: 2.67 px
color_20261024_131431_714_0017.png: 3.62 px
color_20261024_131435_352_0018.png: 0.51 px
color_20261024_131444_512_0019.png: 4.68 px
color_20261024_131447_078_0020.png: 4.58 px

RMS reprojection error: 3.547 px (good: below about 1 px)
Worst image

In [10]:
# Cell 2: draw the detected corners and the board axes on every used image, to check the calibration visually.
# Needs the variables from cell 1 (mtx, dist, rvecs, tvecs, used_image_names, SQUARE_SIZE_M).

DISPLAY_TIME_MS = 500                    # time per image, 0 = wait for a key press
axis_points = np.float32([[3, 0, 0], [0, 3, 0], [0, 0, -3]]) * SQUARE_SIZE_M     # three squares long: x blue, y green, z red (towards the camera)

for i, image_name in enumerate(used_image_names):
    image = cv2.imread(image_name)
    cv2.drawChessboardCorners(image, BOARD_SIZE, imgpoints[i], True)               # the corners that were found
    axis_pixels, _ = cv2.projectPoints(axis_points, rvecs[i], tvecs[i], mtx, dist) # the axes projected with the calibration
    origin = tuple(int(v) for v in imgpoints[i][0].ravel())                        # first corner = board origin
    for pixel, colour in zip(axis_pixels, ((255, 0, 0), (0, 255, 0), (0, 0, 255))):
        image = cv2.line(image, origin, tuple(int(v) for v in pixel.ravel()), colour, 3)
    cv2.imshow("Calibration check", image)
    cv2.waitKey(DISPLAY_TIME_MS)
cv2.destroyAllWindows()

In [8]:
print ('Interinsic Matrix:\n{}'.format(mtx))

rvec1_jacobian = cv2.Rodrigues(rvec1) # convert Jac to Rod -- 3x3 matrix for rotation
rot_ext=rvec1_jacobian[0]

print ('Exterinsic Rotation Matrix:\n{}'.format(rot_ext))

trans_ext=tvec1
print ('Exterinsic Translation Vector:\n{}'.format(trans_ext))

calib_ext=np.hstack((rot_ext, trans_ext))
print ('Exterinsic matrix:\n{}'.format(calib_ext))

Interinsic Matrix:
[[609.411   0.    319.409]
 [  0.    472.444 242.757]
 [  0.      0.      1.   ]]


NameError: name 'rvec1' is not defined